# Pavement Distress Detection — YOLOv11m Training

Trains `yolo11m` on the `Pavement_Distress_1.v1-video_dataset_1.yolov11` dataset.
Designed to run on **Lightning AI** (or any CUDA GPU environment).

**Dataset note:** only ships with `train/`. Section 4 auto-creates an 85/15 train/val split on first run.

**Classes (6):** Cracking, Patches, Pavement_Distress_1, Pothole, Ravelling, Rutting

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics

## 2. Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No CUDA GPU detected.'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

## 3. Configuration

In [ ]:
from pathlib import Path

# EDIT THIS if your dataset is in a different location
DATASET_DIR = Path('Pavement_Distress_1.v1-video_dataset_1.yolov11')
OUTPUT_DIR  = Path('runs')
RUN_NAME    = 'pavement_distress_yolo11m'

MODEL       = 'yolo11m.pt'
EPOCHS      = 100
IMG_SIZE    = 640
BATCH       = -1          # AutoBatch: fills available VRAM
WORKERS     = 8
VAL_SPLIT   = 0.15
SEED        = 42

assert DATASET_DIR.exists(), f'Dataset not found: {DATASET_DIR.resolve()}'
print(f'Dataset: {DATASET_DIR.resolve()}')

## 4. Create train/val split (first run only)

The dataset only has a `train/` folder. This cell moves 15% of images into a new `valid/` folder so YOLO can compute validation metrics. Idempotent — safe to re-run.

In [ ]:
import random
import shutil
import yaml

train_img = DATASET_DIR / 'train' / 'images'
train_lbl = DATASET_DIR / 'train' / 'labels'
val_img   = DATASET_DIR / 'valid' / 'images'
val_lbl   = DATASET_DIR / 'valid' / 'labels'

assert train_img.exists() and train_lbl.exists(), 'train/images or train/labels missing'

if val_img.exists() and any(val_img.iterdir()):
    print(f'Val split already exists at {val_img} — skipping.')
else:
    val_img.mkdir(parents=True, exist_ok=True)
    val_lbl.mkdir(parents=True, exist_ok=True)

    images = sorted(train_img.glob('*.jpg')) + sorted(train_img.glob('*.png'))
    random.seed(SEED)
    random.shuffle(images)
    n_val = int(len(images) * VAL_SPLIT)
    val_images = images[:n_val]

    print(f'Moving {n_val} of {len(images)} images to val split ...')
    for img in val_images:
        shutil.move(str(img), val_img / img.name)
        lbl = train_lbl / f'{img.stem}.txt'
        if lbl.exists():
            shutil.move(str(lbl), val_lbl / lbl.name)
    print('Split done.')

# Rewrite data.yaml with absolute paths
yaml_path = DATASET_DIR / 'data.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['train'] = str(train_img.resolve())
cfg['val']   = str(val_img.resolve())
cfg.pop('test', None)
with open(yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f)

print(f"\nClasses ({cfg['nc']}): {cfg['names']}")
print(f'Train images: {len(list(train_img.glob("*.jpg")))}')
print(f'Val images  : {len(list(val_img.glob("*.jpg")))}')

## 5. Verify all classes have label instances

Catches the silent failure where a class is declared in `data.yaml` but has zero labels (untrainable).

In [ ]:
from collections import Counter

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
names = cfg['names']

per_split = {'train': Counter(), 'valid': Counter()}
for split in per_split:
    lbl_dir = DATASET_DIR / split / 'labels'
    if not lbl_dir.exists():
        continue
    for fp in lbl_dir.glob('*.txt'):
        for line in fp.read_text().splitlines():
            if line.strip():
                per_split[split][int(line.split()[0])] += 1

print(f"{'class':<28}{'train':>10}{'valid':>10}{'total':>10}")
print('-' * 58)
total = Counter()
for cid, name in enumerate(names):
    t = per_split['train'].get(cid, 0)
    v = per_split['valid'].get(cid, 0)
    total[cid] = t + v
    print(f'{cid}: {name:<25}{t:>10}{v:>10}{t+v:>10}')

missing = [names[i] for i in range(len(names)) if total[i] == 0]
if missing:
    print(f'\nWARNING: classes with 0 instances (will not be learned): {missing}')
else:
    print('\nAll classes have instances. Good to train.')

## 6. Train YOLOv11m

Key flags:
- `batch=-1` → AutoBatch picks the largest batch that fits in VRAM
- `amp=True` → FP16 mixed precision (Tensor Cores)
- `cache='ram'` → images held in RAM, eliminates disk I/O bottleneck
- `cos_lr=True` → cosine LR schedule
- `patience=20` → early stop if no improvement for 20 epochs

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    workers=WORKERS,
    device=0,
    amp=True,
    cache='ram',
    optimizer='SGD',
    lr0=0.01,
    cos_lr=True,
    close_mosaic=10,
    patience=20,
    project=str(OUTPUT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    seed=SEED,
)

## 7. Validate on val split

In [ ]:
best = OUTPUT_DIR / RUN_NAME / 'weights' / 'best.pt'
model = YOLO(str(best))
metrics = model.val(data=str(yaml_path), imgsz=IMG_SIZE)
print(f'mAP50   : {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 8. (Optional) Quick inference test

In [ ]:
import glob

samples = glob.glob(str(DATASET_DIR / 'valid' / 'images' / '*.jpg'))[:3]
if samples:
    model = YOLO(str(best))
    res = model.predict(samples, imgsz=IMG_SIZE, conf=0.25, save=True)
    print(f'Saved predictions to: {res[0].save_dir}')